# 从零实现softmax层

引入3.5节中提到的Fashion-mnist数据集，设置batchsize为256


In [1]:
import torch
from IPython import display
from d2l import torch as d2l

batch_size = 256
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)


c:\Users\20249\.conda\envs\test1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 初始化模型参数

因为每个样本都是$ 28 \ times 28 $ 的灰度图，所以每个样本的像素数量为$ 784 $。

$Y = XW + b$ \
其中X为（numsamples， 784）， 在softmax中，权重将会构成一个784*10的矩阵， 偏置项将会构成一个（1， 10）向量。\
在这里使用正态分布初始化权重，偏置项初始化为0

In [2]:
num_inputs = 784
num_outputs = 10
W = torch.normal(0, 0.01, size=(num_inputs, num_outputs), requires_grad=True)
b = torch.zeros(num_outputs,requires_grad=True)

## 定义softmax操作

主要考虑sum函数求和，分为两种情况，一种是保留维度，另一种是不保留维度。

In [3]:
X = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
X.sum(dim=0,keepdim=True), X.sum(dim=1,keepdim=True)

(tensor([[5., 7., 9.]]),
 tensor([[ 6.],
         [15.]]))

In [ ]:
X.sum(dim=0), X.sum(dim=1)  # 这里被直接降维了

(tensor([5., 7., 9.]), tensor([ 6., 15.]))

softmax公式：\
$
\hat{y} = softmax(o) \\
\hat{y}_j = \frac{e^{o_j}}{\sum_{k}e^{o_k}}
$


可以得到：\
$
softmax(X)_{ij} = \frac{exp(X_{ij})}{\sum_{k}exp(X_{ik})}
$

In [5]:
def softmax(X):
    X_exp = torch.exp(X)
    partition = X_exp.sum(dim=1,keepdim=True)
    return X_exp / partition  # 这里用了广播机制，每一行的元素都除以这一行的和
    

## 定义模型

In [6]:
def net(X):
    return softmax(torch.matmul(X.reshape(-1, W.shape[0]), W) + b)


## 定义损失函数

引入交叉熵损失函数：
$$l(\mathbf{y}, \mathbf{\hat{y}}) = - \sum_{j=1}^q y_j \log \hat{y}_j$$
$
\hat{y} 表示模型对所有类别的预测概率分布
$
\
y表示真实标签的 one-hot 编码


创建一个数据样本y_hat, 包括两个样本在三个类别的预测概率，以及他们对应的标签y这样我们就知道这两个样本中哪个是正确的预测

In [8]:
y = torch.tensor([0, 2])
y_hat = torch.tensor([[0.1, 0.3, 0.6], [0.3, 0.2, 0.5]])  # 第一个样本中第一类预测是正确的，第二个样本中第三类预测是正确的
y_hat[[0, 1], y]

tensor([0.1000, 0.5000])

In [10]:
def cross_entropy(y_hat, y):
    return -torch.log(y_hat[range(len(y_hat)), y])  # 只有对正确预测的类别取对数
cross_entropy(y_hat, y)

tensor([2.3026, 0.6931])

## 分类精度

为了计算精度，我们执行以下操作。 首先，如果$\hat{y}$是矩阵，那么假定第二个维度存储每个类的预测分数。 我们使用argmax获得每行中最大元素的索引来获得预测类别。 然后我们将预测类别与真实y元素进行比较。 由于等式运算符“==”对数据类型很敏感， 因此我们将y_hat的数据类型转换为与y的数据类型一致。 结果是一个包含0（错）和1（对）的张量。 最后，我们求和会得到正确预测的数量。

In [ ]:
def accuracy(y_hat, y):  #@save
    """计算预测正确的数量"""
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        y_hat = y_hat.argmax(axis=1)
    cmp = y_hat.type(y.dtype) == y  # 将预测结果的数据类型转换为与真实标签 y 相同（确保类型一致）
    return float(cmp.type(y.dtype).sum())